# CMAPSS LLM Anomaly Detection (Ollama + Open-Source Fallback)

This notebook builds an LLM-style anomaly detector for NASA CMAPSS by:
1. Converting engine-cycle rows to text.
2. Embedding text with an open-source model (`Ollama` or HuggingFace).
3. Training a classifier head and tuning threshold for best F1.

Note: HuggingFace models require internet (unless already cached). Ollama can run fully local after `ollama pull`.

Target: reach high test F1 (try for `>= 0.90`), while being realistic that final score depends on dataset split, threshold, and compute.

In [1]:
# If needed, run once:
!pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm requests sentence-transformers
#For Ollama backend, install Ollama separately and pull a model, for example:


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import shutil
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import requests

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

SEED = 42
DATA_DIR = "/content/drive/MyDrive/data/"
DATASET = "FD001"      # FD001, FD002, FD003, FD004
RUL_THRESHOLD = 30       # anomaly label: RUL <= threshold
VAL_ENGINE_RATIO = 0.2
MIN_SENSOR_VAR = 1e-6

# Backend: "auto", "ollama", "hf"
EMBED_BACKEND = "auto"
OLLAMA_MODEL = "nomic-embed-text"
HF_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# For speed on CPU; set None for full data
MAX_TRAIN_ROWS = None
MAX_VAL_ROWS = None
MAX_TEST_ROWS = None

print(f"Dataset={DATASET}, threshold={RUL_THRESHOLD}, backend={EMBED_BACKEND}")

Dataset=FD001, threshold=30, backend=auto


In [8]:
@dataclass
class CmapssData:
    train: pd.DataFrame
    test: pd.DataFrame
    features: list


def load_cmapss(dataset="FD001", data_dir="data", rul_threshold=30, min_sensor_var=1e-6):
    train_path = os.path.join(data_dir, f"train_{dataset}.csv")
    test_path = os.path.join(data_dir, f"test_{dataset}.csv")
    rul_path = os.path.join(data_dir, f"RUL_{dataset}.txt")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    rul_df = pd.read_csv(rul_path, header=None, names=["final_rul"])

    # Drop empty columns from CSV conversion
    train_df = train_df.dropna(axis=1, how="all").copy()
    test_df = test_df.dropna(axis=1, how="all").copy()

    # Train RUL
    max_cycle_train = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")
    train_df = train_df.merge(max_cycle_train, on="unit_number", how="left")
    train_df["RUL"] = train_df["max_cycle"] - train_df["time_in_cycles"]
    train_df = train_df.drop(columns=["max_cycle"])

    # Test row-level RUL: final_rul(engine) + remaining cycles from row to end-of-sequence
    unit_order = np.sort(test_df["unit_number"].unique())
    if len(unit_order) != len(rul_df):
        raise ValueError("Mismatch between test engines and RUL file length.")

    final_rul_map = dict(zip(unit_order, rul_df["final_rul"].astype(float).tolist()))
    test_df["final_rul"] = test_df["unit_number"].map(final_rul_map)
    max_cycle_test = test_df.groupby("unit_number")["time_in_cycles"].transform("max")
    test_df["RUL"] = test_df["final_rul"] + (max_cycle_test - test_df["time_in_cycles"])

    # Binary labels
    train_df["anomaly"] = (train_df["RUL"] <= rul_threshold).astype(int)
    test_df["anomaly"] = (test_df["RUL"] <= rul_threshold).astype(int)

    op_cols = [c for c in train_df.columns if c.startswith("operational_setting_")]
    sensor_cols = [c for c in train_df.columns if c.startswith("sensor_measurement_")]
    sensor_cols = [c for c in sensor_cols if train_df[c].var() > min_sensor_var]

    base_features = op_cols + sensor_cols

    def add_engineered(df, features):
        out = df.copy()
        created = []
        for c in features:
            rmean = f"{c}_rmean5"
            delta = f"{c}_delta"
            out[rmean] = out.groupby("unit_number")[c].transform(lambda s: s.rolling(5, min_periods=1).mean())
            out[delta] = out.groupby("unit_number")[c].transform(lambda s: s - s.iloc[0])
            created.extend([rmean, delta])
        return out, features + created

    train_df, all_features = add_engineered(train_df, base_features)
    test_df, _ = add_engineered(test_df, base_features)

    return CmapssData(train=train_df, test=test_df, features=all_features)


def row_to_text(df, feature_cols, precision=5):
    # Keep text compact; this helps embedding quality and speed.
    data = df[feature_cols].fillna(0.0).to_numpy()
    names = np.array(feature_cols)
    texts = []
    for row in data:
        parts = [f"{n}={v:.{precision}f}" for n, v in zip(names, row)]
        texts.append("engine_state " + " ".join(parts))
    return texts


def sample_rows(df, max_rows=None, seed=42):
    if max_rows is None or len(df) <= max_rows:
        return df
    return df.sample(n=max_rows, random_state=seed).sort_index()


In [9]:
ds = load_cmapss(dataset=DATASET, data_dir=DATA_DIR, rul_threshold=RUL_THRESHOLD, min_sensor_var=MIN_SENSOR_VAR)

train_df = ds.train.copy()
test_df = ds.test.copy()
feature_cols = ds.features

print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Feature count:", len(feature_cols))
print("Train anomaly ratio:", round(train_df['anomaly'].mean(), 4))
print("Test anomaly ratio:", round(test_df['anomaly'].mean(), 4))

# Engine-level split to reduce leakage.
engine_labels = train_df.groupby("unit_number")["anomaly"].max().reset_index()
train_units, val_units = train_test_split(
    engine_labels["unit_number"].values,
    test_size=VAL_ENGINE_RATIO,
    random_state=SEED,
    stratify=engine_labels["anomaly"].values,
)

train_part = train_df[train_df["unit_number"].isin(train_units)].copy()
val_part = train_df[train_df["unit_number"].isin(val_units)].copy()

test_last = test_df.sort_values(["unit_number", "time_in_cycles"]).groupby("unit_number").tail(1).copy()

train_part = sample_rows(train_part, MAX_TRAIN_ROWS, seed=SEED)
val_part = sample_rows(val_part, MAX_VAL_ROWS, seed=SEED)
test_rows_eval = sample_rows(test_df, MAX_TEST_ROWS, seed=SEED)

X_train_text = row_to_text(train_part, feature_cols)
X_val_text = row_to_text(val_part, feature_cols)
X_test_last_text = row_to_text(test_last, feature_cols)
X_test_rows_text = row_to_text(test_rows_eval, feature_cols)

y_train = train_part["anomaly"].astype(int).values
y_val = val_part["anomaly"].astype(int).values
y_test_last = test_last["anomaly"].astype(int).values
y_test_rows = test_rows_eval["anomaly"].astype(int).values

print("Split sizes -> train:", len(y_train), "val:", len(y_val), "test_last:", len(y_test_last), "test_rows:", len(y_test_rows))

Train rows: 20631  Test rows: 13096
Feature count: 54
Train anomaly ratio: 0.1503
Test anomaly ratio: 0.0254
Split sizes -> train: 16340 val: 4291 test_last: 100 test_rows: 13096


In [10]:
def embed_ollama(texts, model="nomic-embed-text", batch_size=128, host="http://localhost:11434"):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        # Newer Ollama API
        r = requests.post(
            f"{host}/api/embed",
            json={"model": model, "input": batch},
            timeout=120,
        )
        if r.status_code == 200 and "embeddings" in r.json():
            vecs = r.json()["embeddings"]
            all_vecs.extend(vecs)
            continue

        # Backward-compatible fallback
        for text in batch:
            r2 = requests.post(
                f"{host}/api/embeddings",
                json={"model": model, "prompt": text},
                timeout=120,
            )
            r2.raise_for_status()
            all_vecs.append(r2.json()["embedding"])

    return np.asarray(all_vecs, dtype=np.float32)


def embed_hf(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=128):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vecs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vecs.astype(np.float32)


def embed_auto(texts, backend="auto"):
    if backend in ("auto", "ollama"):
        ollama_available = shutil.which("ollama") is not None
        if backend == "ollama" and not ollama_available:
            raise RuntimeError("Ollama CLI not found. Install Ollama or switch backend='hf'.")
        if ollama_available:
            try:
                print(f"Embedding with Ollama model: {OLLAMA_MODEL}")
                return embed_ollama(texts, model=OLLAMA_MODEL)
            except Exception as e:
                if backend == "ollama":
                    raise
                print(f"Ollama failed, fallback to HF: {e}")

    print(f"Embedding with HF model: {HF_MODEL}")
    return embed_hf(texts, model_name=HF_MODEL)


X_train_emb = embed_auto(X_train_text, backend=EMBED_BACKEND)
X_val_emb = embed_auto(X_val_text, backend=EMBED_BACKEND)
X_test_last_emb = embed_auto(X_test_last_text, backend=EMBED_BACKEND)
X_test_rows_emb = embed_auto(X_test_rows_text, backend=EMBED_BACKEND)

print("Embedding shapes:")
print("train", X_train_emb.shape, "val", X_val_emb.shape, "test_last", X_test_last_emb.shape)

Embedding with HF model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Embedding with HF model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shapes:
train (16340, 384) val (4291, 384) test_last (100, 384)


In [11]:
def best_threshold_by_f1(y_true, y_score):
    p, r, th = precision_recall_curve(y_true, y_score)
    f1 = (2 * p * r) / (p + r + 1e-12)
    if len(th) == 0:
        return 0.5, 0.0
    idx = int(np.nanargmax(f1[:-1]))
    return float(th[idx]), float(f1[idx])


def evaluate_binary(y_true, y_pred, title):
    print(f"\n{title}")
    print("F1:", round(f1_score(y_true, y_pred), 4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=["normal", "anomaly"], digits=4))


candidates = {
    "logreg": LogisticRegression(max_iter=4000, class_weight="balanced", random_state=SEED),
    "rf": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=1,
    ),
}

best = None
for name, model in candidates.items():
    model.fit(X_train_emb, y_train)
    val_score = model.predict_proba(X_val_emb)[:, 1]
    th, val_f1 = best_threshold_by_f1(y_val, val_score)
    print(f"{name}: best val F1={val_f1:.4f} at threshold={th:.4f}")

    if best is None or val_f1 > best["val_f1"]:
        best = {"name": name, "model": model, "threshold": th, "val_f1": val_f1}

print("\nSelected model:", best["name"])

best_model = best["model"]
best_th = best["threshold"]

pred_test_last = (best_model.predict_proba(X_test_last_emb)[:, 1] >= best_th).astype(int)
pred_test_rows = (best_model.predict_proba(X_test_rows_emb)[:, 1] >= best_th).astype(int)

evaluate_binary(y_test_last, pred_test_last, "Test (last cycle per engine)")
evaluate_binary(y_test_rows, pred_test_rows, "Test (all available cycles)")

last_f1 = f1_score(y_test_last, pred_test_last)
if last_f1 >= 0.90:
    print(f"Target reached on last-cycle test: F1={last_f1:.4f}")
else:
    print(f"Last-cycle test F1={last_f1:.4f}. Try threshold sweep, larger embedding model, or more feature engineering.")

logreg: best val F1=0.5822 at threshold=0.5477
rf: best val F1=0.6029 at threshold=0.2020

Selected model: rf

Test (last cycle per engine)
F1: 0.5614
Confusion matrix:
 [[59 16]
 [ 9 16]]
              precision    recall  f1-score   support

      normal     0.8676    0.7867    0.8252        75
     anomaly     0.5000    0.6400    0.5614        25

    accuracy                         0.7500       100
   macro avg     0.6838    0.7133    0.6933       100
weighted avg     0.7757    0.7500    0.7592       100


Test (all available cycles)
F1: 0.2495
Confusion matrix:
 [[11763  1001]
 [  142   190]]
              precision    recall  f1-score   support

      normal     0.9881    0.9216    0.9537     12764
     anomaly     0.1595    0.5723    0.2495       332

    accuracy                         0.9127     13096
   macro avg     0.5738    0.7469    0.6016     13096
weighted avg     0.9671    0.9127    0.9358     13096

Last-cycle test F1=0.5614. Try threshold sweep, larger embedding mo